# AutoDub Studio - Neural Voice Lab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syedj7895-cell/ai-video-dubber-AutoDub-Studio-Automatic-Dubbing-Engine-ElevenLabs-Quality-Open-Source-/blob/main/TTS_Tester.ipynb)

Full-bleed ElevenLabs-grade studio UI for comparing **16 free, open-weight TTS engines** - tuned for **Hindi dubbing**.

- Engines: Edge-TTS, gTTS, MMS, Piper, Kokoro, IndicF5, F5-TTS, Chatterbox, XTTS v2, CosyVoice 2/3, VibeVoice, VibeVoice-Hindi-7B, Veena, VEXYL, Qwen3-TTS
- Weights download **only when you press Generate** in that model tab
- Fail-soft: if an engine/weights cannot be provisioned it falls back to a reliable neural voice and says so in the diagnostics panel
- No API keys, no billing


In [ ]:
# ── 1 ▸ Environment bootstrap & core dependencies ────────────────────
import os, pathlib, subprocess, sys

REPO_URL = ("https://github.com/syedj7895-cell/"
            "ai-video-dubber-AutoDub-Studio-Automatic-Dubbing-Engine-"
            "ElevenLabs-Quality-Open-Source-.git")

print("⚡ Warming up core audio packages ...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "gradio>=4.44", "soundfile", "librosa", "edge-tts", "gTTS",
                "transformers", "accelerate", "scipy", "indic-transliteration"],
               check=False)

if not pathlib.Path("tts_tester.py").exists():
    if not pathlib.Path("ai-video-dubber").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "ai-video-dubber"], check=False)
    if pathlib.Path("ai-video-dubber").exists():
        os.chdir("ai-video-dubber")

sys.path.insert(0, ".")
if pathlib.Path("/content/CosyVoice").is_dir():
    sys.path.insert(0, "/content/CosyVoice")
    sys.path.insert(0, "/content/CosyVoice/third_party/Matcha-TTS")

import tts_tester
print("✅ TTS Tester engine active ·", len(tts_tester.BACKENDS), "backends registered.")


In [ ]:
# ── 2 ▸ ElevenLabs-grade Studio UI (full-bleed, 16:9) ─────────────────
import inspect
import gradio as gr
import numpy as np
import soundfile as sf
import tempfile
import tts_tester
from tts_tester import BACKENDS, get_backend

CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800&display=swap');

:root{
  --el-bg:#0a0a0b; --el-panel:#111113; --el-panel-2:#16161a;
  --el-border:#232329; --el-border-2:#2e2e36;
  --el-text:#f3f3f5; --el-muted:#8b8b94; --el-dim:#5f5f68;
  --el-radius:14px;
}

html, body, .gradio-container { background:var(--el-bg) !important; }
body, .gradio-container, .gradio-container * {
  font-family:'Inter', -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif !important;
  -webkit-font-smoothing:antialiased;
}
.gradio-container{ max-width:100% !important; width:100% !important;
  margin:0 !important; padding:0 !important; color:var(--el-text) !important; }
.gradio-container .prose, .gradio-container p { color:var(--el-text) !important; }
footer, .footer, .built-with, .show-api { display:none !important; }

/* ── Top navigation bar ─────────────────────────────────────────────── */
.el-nav{ position:sticky; top:0; z-index:80; display:flex; align-items:center;
  gap:30px; height:62px; padding:0 34px; background:rgba(10,10,11,.88);
  backdrop-filter:blur(18px) saturate(160%); border-bottom:1px solid var(--el-border); }
.el-brand{ display:flex; align-items:center; gap:11px; font-size:16px;
  font-weight:700; letter-spacing:-.025em; color:#fff; white-space:nowrap; }
.el-logo{ width:27px; height:27px; border-radius:8px; flex:0 0 27px;
  background:linear-gradient(135deg,#ffffff,#b4b4c2); color:#0a0a0b;
  display:flex; align-items:center; justify-content:center;
  font-weight:800; font-size:14px; letter-spacing:-.05em; }
.el-links{ display:flex; gap:24px; font-size:13.5px; color:var(--el-muted);
  font-weight:500; }
.el-links span{ cursor:pointer; transition:color .15s ease; }
.el-links span:hover{ color:#fff; }
.el-nav-right{ margin-left:auto; display:flex; align-items:center; gap:12px; }
.el-pill{ padding:7px 15px; border-radius:99px; font-size:12.5px; font-weight:600;
  background:var(--el-panel-2); color:var(--el-text);
  border:1px solid var(--el-border-2); white-space:nowrap; }
.el-pill.solid{ background:#fff; color:#0a0a0b; border-color:#fff; }
.el-pill.green{ background:rgba(16,185,129,.13); color:#34d399;
  border-color:rgba(16,185,129,.32); }
"""

CSS += """
/* ── 16:9 hero stage ────────────────────────────────────────────────── */
.el-hero{ position:relative; width:100%; aspect-ratio:16 / 9; max-height:44vh;
  min-height:270px; overflow:hidden; display:flex; flex-direction:column;
  justify-content:flex-end; padding:0 60px 46px; border-bottom:1px solid var(--el-border);
  background:
    radial-gradient(1100px 560px at 18% 4%, rgba(99,102,241,.20), transparent 62%),
    radial-gradient(900px 520px at 88% 96%, rgba(236,72,153,.15), transparent 64%),
    radial-gradient(700px 420px at 55% 50%, rgba(56,189,248,.09), transparent 70%),
    linear-gradient(180deg,#0d0d11 0%, #0a0a0b 100%); }
.el-wave{ position:absolute; inset:auto 0 0 0; height:132px; display:flex;
  align-items:flex-end; gap:4px; padding:0 60px; opacity:.30; }
.el-wave i{ flex:1 1 auto; background:linear-gradient(180deg,#7c8cf8,#4f46e5);
  border-radius:3px 3px 0 0; animation:elwave 2.6s ease-in-out infinite; }
@keyframes elwave{ 0%,100%{ transform:scaleY(.24);} 50%{ transform:scaleY(1);} }
.el-hero-inner{ position:relative; z-index:2; }
.el-kicker{ display:inline-flex; align-items:center; gap:8px; font-size:11.5px;
  font-weight:600; letter-spacing:.14em; text-transform:uppercase;
  color:#a5b4fc; margin-bottom:15px; }
.el-hero h1{ margin:0; font-size:clamp(30px, 3.5vw, 54px); font-weight:800;
  letter-spacing:-.035em; line-height:1.03; color:#fff; }
.el-hero h1 em{ font-style:normal;
  background:linear-gradient(96deg,#ffffff 18%,#a5b4fc 62%,#f0abfc 100%);
  -webkit-background-clip:text; -webkit-text-fill-color:transparent; }
.el-hero p{ margin:14px 0 0; max-width:760px; color:var(--el-muted);
  font-size:15px; line-height:1.62; }
.el-hero-tags{ display:flex; flex-wrap:wrap; gap:9px; margin-top:22px; }

/* ── Layout shell ───────────────────────────────────────────────────── */
.el-wrap{ padding:26px 34px 70px !important; }
.el-panel{ background:var(--el-panel) !important;
  border:1px solid var(--el-border) !important;
  border-radius:var(--el-radius) !important; padding:20px !important; }

/* ── Model tab rail (horizontally scrollable, ElevenLabs nav style) ─── */
div[role="tablist"], .tab-nav{ display:flex !important; flex-wrap:nowrap !important;
  overflow-x:auto !important; gap:2px !important; background:transparent !important;
  border-bottom:1px solid var(--el-border) !important; padding-bottom:0 !important;
  scrollbar-width:thin; }
div[role="tablist"]::-webkit-scrollbar{ height:4px; }
div[role="tablist"]::-webkit-scrollbar-thumb{ background:#2c2c34; border-radius:99px; }
button[role="tab"]{ background:transparent !important; border:none !important;
  border-bottom:2px solid transparent !important; color:var(--el-muted) !important;
  font-size:13px !important; font-weight:500 !important; padding:11px 15px !important;
  border-radius:0 !important; white-space:nowrap !important;
  transition:color .15s ease, border-color .15s ease !important; }
button[role="tab"]:hover{ color:#d8d8de !important; }
button[role="tab"][aria-selected="true"]{ color:#fff !important;
  border-bottom:2px solid #fff !important; font-weight:600 !important; }
"""

CSS += """
/* ── Voice identity card ────────────────────────────────────────────── */
.el-voice{ display:flex; gap:16px; align-items:flex-start; background:var(--el-panel);
  border:1px solid var(--el-border); border-radius:var(--el-radius);
  padding:18px 20px; margin:18px 0 6px; }
.el-avatar{ width:52px; height:52px; flex:0 0 52px; border-radius:13px;
  display:flex; align-items:center; justify-content:center; font-weight:700;
  font-size:17px; color:#fff; letter-spacing:-.03em;
  box-shadow:0 6px 20px rgba(0,0,0,.45); }
.el-vname{ display:flex; align-items:center; gap:9px; font-size:17px;
  font-weight:600; color:#fff; letter-spacing:-.02em; }
.el-vsub{ color:var(--el-muted); font-size:13px; margin-top:5px; line-height:1.55; }
.el-tag{ display:inline-flex; align-items:center; gap:6px; padding:4px 11px;
  border-radius:99px; background:#191920; border:1px solid var(--el-border-2);
  color:#b6b6c0; font-size:11.5px; font-weight:500; }
.el-tag.accent{ background:rgba(99,102,241,.14); border-color:rgba(99,102,241,.34);
  color:#a5b4fc; }
.el-tag.ok{ background:rgba(16,185,129,.13); border-color:rgba(16,185,129,.3);
  color:#34d399; }
.el-meta{ display:flex; flex-wrap:wrap; gap:8px; margin-top:12px; }
.el-link{ color:#8ab4ff; font-size:11.5px; text-decoration:none; font-weight:500; }
.el-link:hover{ text-decoration:underline; }

/* ── Section headings ───────────────────────────────────────────────── */
.el-h{ display:flex; align-items:center; gap:9px; font-size:12px; font-weight:600;
  letter-spacing:.11em; text-transform:uppercase; color:var(--el-dim);
  margin:0 0 12px; }

/* ── Buttons (ElevenLabs pill / primary CTA) ────────────────────────── */
.el-cta button{ background:#ffffff !important; color:#0a0a0b !important;
  border:none !important; border-radius:10px !important; font-weight:700 !important;
  font-size:14.5px !important; height:46px !important; letter-spacing:-.01em !important;
  box-shadow:0 8px 26px rgba(255,255,255,.10) !important;
  transition:transform .15s ease, box-shadow .15s ease !important; }
.el-cta button:hover{ transform:translateY(-1px) !important;
  box-shadow:0 12px 34px rgba(255,255,255,.17) !important; }
.el-ghost button{ background:var(--el-panel-2) !important; color:var(--el-text) !important;
  border:1px solid var(--el-border-2) !important; border-radius:10px !important;
  font-weight:600 !important; font-size:13.5px !important; height:42px !important;
  transition:border-color .15s ease, background .15s ease !important; }
.el-ghost button:hover{ background:#1d1d24 !important; border-color:#3a3a44 !important; }
.el-copy button{ background:transparent !important; color:var(--el-muted) !important;
  border:1px solid var(--el-border) !important; border-radius:9px !important;
  font-size:12px !important; height:34px !important; }
.el-copy button:hover{ color:#fff !important; border-color:#3a3a44 !important; }

/* ── Console / diagnostics ──────────────────────────────────────────── */
.el-console textarea{ background:#08080a !important; color:#c7f5dc !important;
  font-family:ui-monospace, SFMono-Regular, Menlo, Consolas, monospace !important;
  font-size:12px !important; line-height:1.6 !important;
  border:1px solid #1d1d23 !important; border-radius:12px !important;
  padding:14px !important; }

/* ── Inputs, dropdowns, sliders ─────────────────────────────────────── */
.gradio-container textarea, .gradio-container input[type="text"],
.gradio-container input[type="number"]{
  background:#121217 !important; color:var(--el-text) !important;
  border:1px solid var(--el-border) !important; border-radius:10px !important;
  font-size:13.5px !important; }
.gradio-container textarea:focus, .gradio-container input:focus{
  border-color:#4b4b58 !important; box-shadow:0 0 0 3px rgba(99,102,241,.13) !important; }
.gradio-container label > span, .gradio-container .block-title span{
  color:var(--el-muted) !important; font-size:11.5px !important;
  font-weight:600 !important; letter-spacing:.05em !important;
  text-transform:uppercase !important; }
.gradio-container input[type="range"]{ accent-color:#ffffff !important; }
.gradio-container .wrap, .gradio-container .container{ background:transparent !important; }
.gradio-container .form, .gradio-container .panel, .gradio-container .block{
  background:transparent !important; border:none !important; }
.gradio-container .block{ padding:0 !important; }
.gradio-container .audio-player audio{ width:100% !important; border-radius:10px !important; }

/* ── Responsive: keep the widescreen feel on laptops ────────────────── */
@media (max-width: 1180px){
  .el-links{ display:none; }
  .el-hero{ padding:0 34px 34px; min-height:230px; }
  .el-wave{ padding:0 34px; height:96px; }
  .el-wrap{ padding:20px 20px 60px !important; }
}
"""

# Gradio applies ``elem_classes`` to the component root, so style the bare
# class on the <button> element itself (the descendant rules above are kept
# for wrapper-based usage).
CSS += """
button.el-cta{ background:#ffffff !important; color:#0a0a0b !important;
  border:none !important; border-radius:10px !important; font-weight:700 !important;
  font-size:14.5px !important; letter-spacing:-.01em !important;
  box-shadow:0 8px 26px rgba(255,255,255,.10) !important;
  transition:transform .15s ease, box-shadow .15s ease !important; }
button.el-cta:hover{ transform:translateY(-1px) !important;
  box-shadow:0 12px 34px rgba(255,255,255,.17) !important; }
button.el-ghost{ background:var(--el-panel-2) !important; color:var(--el-text) !important;
  border:1px solid var(--el-border-2) !important; border-radius:10px !important;
  font-weight:600 !important; font-size:13.5px !important;
  transition:border-color .15s ease, background .15s ease !important; }
button.el-ghost:hover{ background:#1d1d24 !important; border-color:#3a3a44 !important; }
button.el-copy{ background:transparent !important; color:var(--el-muted) !important;
  border:1px solid var(--el-border) !important; border-radius:9px !important;
  font-size:12px !important; }
button.el-copy:hover{ color:#fff !important; border-color:#3a3a44 !important; }
"""

JS_COPY_CONSOLE = """
(text) => {
    if (!text) return;
    if (navigator.clipboard && window.isSecureContext) {
        navigator.clipboard.writeText(text);
    } else {
        const ta = document.createElement('textarea');
        ta.value = text;
        ta.style.position = 'fixed';
        ta.style.opacity = '0';
        document.body.appendChild(ta);
        ta.select();
        document.execCommand('copy');
        document.body.removeChild(ta);
    }
}
"""

_WAVE_BARS = "".join(
    f'<i style="animation-delay:{round(i * 0.045, 3)}s"></i>' for i in range(56)
)


def _brand_bar(model_count: int) -> str:
    return f'''
    <div class="el-nav">
      <div class="el-brand"><span class="el-logo">II</span>AutoDub Studio</div>
      <div class="el-links">
        <span>Voices</span><span>Studio</span><span>Dubbing</span>
        <span>Models</span><span>Docs</span>
      </div>
      <div class="el-nav-right">
        <span class="el-pill green">{model_count} engines online</span>
        <span class="el-pill">Free · Open source</span>
        <span class="el-pill solid">Launch console</span>
      </div>
    </div>'''


def _hero_html() -> str:
    return f'''
    <div class="el-hero">
      <div class="el-wave">{_WAVE_BARS}</div>
      <div class="el-hero-inner">
        <div class="el-kicker">Automatic dubbing engine · Hindi / English</div>
        <h1>Neural voice studio for<br><em>flawless dubbing</em></h1>
        <p>Sixteen open-weight speech engines in one console. Clone a voice,
           dial in the emotion, and render broadcast-ready Hindi dialogue —
           no API keys, no per-character billing.</p>
        <div class="el-hero-tags">
          <span class="el-tag accent">Zero-shot cloning</span>
          <span class="el-tag">Hindi native</span>
          <span class="el-tag">Emotion aware</span>
          <span class="el-tag">Demucs + Pyannote</span>
        </div>
      </div>
    </div>'''


def _initials(name: str) -> str:
    parts = [p for p in name.replace("-", " ").split() if p]
    return "".join(p[0] for p in parts[:2]).upper() or "AI"


def _voice_card(info) -> str:
    clone = ('<span class="el-tag accent">Voice cloning</span>' if info.clone
             else '<span class="el-tag">Fixed neural voice</span>')
    return f'''
    <div class="el-voice">
      <div class="el-avatar" style="background:{info.avatar_color}">{_initials(info.name)}</div>
      <div style="flex:1 1 auto">
        <div class="el-vname">{info.name}
          <span class="el-tag">{info.category}</span></div>
        <div class="el-vsub">{info.notes}</div>
        <div class="el-meta">
          <span class="el-tag">by {info.creator}</span>
          <span class="el-tag">{info.languages}</span>
          {clone}
          <span class="el-tag">{info.size}</span>
          <span class="el-tag">{info.license}</span>
          <a class="el-link" href="{info.repo}" target="_blank">Weights &amp; docs</a>
        </div>
      </div>
    </div>'''


def _fmt_log(lines) -> str:
    return "\n".join(str(line) for line in lines)


def make_load_fn(bid, con_lines):
    def _load():
        b = get_backend(bid)
        con_lines.clear()
        con_lines.append(f"[{b.info.name}] Preparing engine ...")
        try:
            b.load(lambda m: con_lines.append(str(m)))
            con_lines.append("Engine ready — press Generate speech.")
        except Exception as e:
            con_lines.append(f"Load failed: {e}")
        return _fmt_log(con_lines)
    return _load


def make_synth_fn(bid, con_lines, settings_names):
    def _synth(text, voice, language, ref, *sargs):
        b = get_backend(bid)
        con_lines.clear()
        con_lines.append(f"[{b.info.name}] {text[:70]!r}")
        settings = dict(zip(settings_names, sargs))
        if not b.loaded:
            con_lines.append("Auto-loading engine into memory ...")
            try:
                b.load(lambda m: con_lines.append(str(m)))
            except Exception as e:
                con_lines.append(f"Load failed: {e}")
                return _fmt_log(con_lines), None
        try:
            y, sr = b.synthesize(text, voice=voice, language=language,
                                 ref_audio=ref, settings=settings,
                                 log=lambda m: con_lines.append(str(m)))
            y = np.asarray(y, dtype=np.float32)
            p = tempfile.mkstemp(suffix=".wav")[1]
            sf.write(p, y, int(sr))
            con_lines.append(f"Rendered {len(y) / sr:.2f}s of speech at {sr} Hz.")
            return _fmt_log(con_lines), p
        except Exception as e:
            con_lines.append(f"Synthesis failed: {e}")
            return _fmt_log(con_lines), None
    return _synth


# Gradio 6 moved ``css`` from the Blocks constructor to launch(); Gradio 4/5
# only accept it on Blocks. Wire it up for whichever version Colab installs.
_BLOCKS_KW = {"title": "AutoDub Studio · Neural Voice Lab"}
_LAUNCH_KW = {"share": True, "debug": False}
if "css" in inspect.signature(gr.Blocks.__init__).parameters:
    _BLOCKS_KW["css"] = CSS
if "css" in inspect.signature(gr.Blocks.launch).parameters:
    _LAUNCH_KW["css"] = CSS

with gr.Blocks(**_BLOCKS_KW) as demo:
    gr.HTML(_brand_bar(len(BACKENDS)))
    gr.HTML(_hero_html())

    with gr.Column(elem_classes=["el-wrap"]):
        with gr.Tabs():
            for bid, cls in BACKENDS.items():
                info = cls.info
                inst = cls()
                con_lines = [f"Ready: {info.name}"]
                with gr.Tab(info.name):
                    gr.HTML(_voice_card(info))
                    with gr.Row():
                        with gr.Column(scale=7, elem_classes=["el-panel"]):
                            gr.HTML('<div class="el-h">Script editor</div>')
                            text_c = gr.Textbox(
                                show_label=False, lines=7, max_lines=18,
                                placeholder="Type or paste the dialogue to synthesize ...",
                                value="नमस्ते! आपका स्वागत है। यह आवाज़ का परीक्षण है।")
                            with gr.Row():
                                if inst.voices():
                                    voice_c = gr.Dropdown(
                                        choices=inst.voices(), label="Voice",
                                        value=inst.voices()[0])
                                else:
                                    voice_c = gr.Dropdown(
                                        choices=[], label="Voice", visible=False)
                                lang_c = gr.Dropdown(
                                    choices=inst.languages(), label="Language",
                                    value=inst.languages()[0])
                            ref_c = gr.Audio(
                                label="Reference clip for voice cloning",
                                type="filepath", visible=info.clone)
                            settings_cs = []
                            settings_names = []
                            schema = inst.settings_schema()
                            if schema:
                                gr.HTML('<div class="el-h" style="margin-top:14px">'
                                        'Voice settings</div>')
                                for s in schema:
                                    if s["type"] == "slider":
                                        settings_cs.append(gr.Slider(
                                            minimum=s.get("min", 0),
                                            maximum=s.get("max", 2),
                                            value=s.get("value", 0),
                                            label=s["label"]))
                                    else:
                                        settings_cs.append(gr.Textbox(
                                            label=s["label"],
                                            value=s.get("value", "")))
                                    settings_names.append(s["name"])
                            with gr.Row():
                                load_btn = gr.Button("Load engine",
                                                     elem_classes=["el-ghost"])
                                synth_btn = gr.Button("Generate speech",
                                                      elem_classes=["el-cta"])
                        with gr.Column(scale=5, elem_classes=["el-panel"]):
                            gr.HTML('<div class="el-h">Rendered output</div>')
                            audio_out = gr.Audio(label="Speech master",
                                                 type="filepath")
                            gr.HTML('<div class="el-h" style="margin-top:18px">'
                                    'Engine diagnostics</div>')
                            con_c = gr.Textbox(lines=12, max_lines=24,
                                               interactive=False,
                                               show_label=False,
                                               elem_classes=["el-console"])
                            copy_btn = gr.Button("Copy log",
                                                 elem_classes=["el-copy"])
                    copy_btn.click(fn=None, inputs=[con_c], js=JS_COPY_CONSOLE)
                    load_btn.click(make_load_fn(bid, con_lines),
                                   inputs=None, outputs=[con_c])
                    synth_btn.click(
                        make_synth_fn(bid, con_lines, settings_names),
                        inputs=[text_c, voice_c, lang_c, ref_c, *settings_cs],
                        outputs=[con_c, audio_out])

demo.launch(**_LAUNCH_KW)
